# Baseline модель

Простая Ridge-регрессия без feature engineering - точка отсчёта для экспериментов.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from src.preprocessing.clean import clean_all
from src.models.evaluate import split_data, regression_metrics, print_metrics

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

## Данные - только числовые признаки без feature engineering

In [ ]:
df = clean_all()
print(f'Shape: {df.shape}')

BASELINE_FEATURES = ['storage_gb', 'ram_gb', 'screen_inch', 'is_laptop']
BASELINE_FEATURES = ['storage_gb', 'ram_gb', 'screen_inch']

# Добавляем is_laptop вручную (без feature engineering)
df['is_laptop'] = (df['device_type'] == 'laptop').astype(int)
BASELINE_FEATURES = ['storage_gb', 'ram_gb', 'screen_inch', 'is_laptop']

X = df[BASELINE_FEATURES]
y = df['price']

X_train, X_val, X_test, y_train, y_val, y_test = split_data(X, y, random_state=RANDOM_SEED)
print(f'Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}')

## Обучение Ridge baseline

In [ ]:
baseline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('model', Ridge(random_state=RANDOM_SEED)),
])
baseline.fit(X_train, y_train)

m_val = regression_metrics(y_val, baseline.predict(X_val))
m_test = regression_metrics(y_test, baseline.predict(X_test))
print_metrics('Ridge Baseline - Val ', m_val)
print_metrics('Ridge Baseline - Test', m_test)

## Анализ ошибок

In [ ]:
y_pred = baseline.predict(X_val)
residuals = y_val.values - y_pred

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(y_pred, residuals, alpha=0.3, s=8)
axes[0].axhline(0, color='red', lw=1)
axes[0].set_xlabel('Предсказание')
axes[0].set_ylabel('Остаток')
axes[0].set_title('Остатки vs предсказание')

axes[1].hist(residuals, bins=50)
axes[1].set_xlabel('Остаток')
axes[1].set_title('Распределение остатков')
plt.tight_layout()

In [ ]:
# Сохраняем результат для сравнения в 03_experiments
baseline_result = {'model': 'Ridge Baseline', **m_val}
print('Baseline result:', baseline_result)